In [0]:
from pyspark.sql.functions import (
    col,
    when,
    concat_ws,
    length,
    current_timestamp
)

# Checkpoint locations
checkpoint_base = (
    "/Volumes/fraud_detection/bronze/"
    "realtime_files/checkpoints"
)

valid_checkpoint = (
    f"{checkpoint_base}/silver_valid_transactions"
)

quarantine_checkpoint = (
    f"{checkpoint_base}/quarantine_transactions"
)

# Read new Bronze records as a stream
bronze_stream = spark.readStream.table(
    "fraud_detection.bronze.realtime_transactions"
)

# Apply data-quality rules
validated_stream = (
    bronze_stream
    .withColumn(
        "validation_errors",
        concat_ws(
            ", ",
            when(
                col("transaction_id").isNull(),
                "missing_transaction_id"
            ),
            when(
                col("card_id").isNull(),
                "missing_card_id"
            ),
            when(
                col("customer_id").isNull(),
                "missing_customer_id"
            ),
            when(
                col("merchant_id").isNull(),
                "missing_merchant_id"
            ),
            when(
                col("amount").isNull() |
                (col("amount") <= 0),
                "invalid_amount"
            ),
            when(
                col("transaction_timestamp").isNull(),
                "invalid_timestamp"
            ),
            when(
                col("_rescued_data").isNotNull(),
                "unexpected_schema"
            )
        )
    )
    .withColumn(
        "has_error",
        length(col("validation_errors")) > 0
    )
    .withColumn(
        "validation_timestamp",
        current_timestamp()
    )
)

# Valid records
valid_stream = (
    validated_stream
    .filter(col("has_error") == False)
    .drop(
        "has_error",
        "validation_errors",
        "_rescued_data"
    )
    .withWatermark(
        "transaction_timestamp",
        "1 day"
    )
    .dropDuplicates(["transaction_id"])
)

# Invalid records
invalid_stream = (
    validated_stream
    .filter(col("has_error") == True)
)

# Write valid records to Silver
valid_query = (
    valid_stream.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        valid_checkpoint
    )
    .trigger(availableNow=True)
    .toTable(
        "fraud_detection.silver.realtime_transactions"
    )
)

valid_query.awaitTermination()

# Write invalid records to Quarantine
invalid_query = (
    invalid_stream.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        quarantine_checkpoint
    )
    .trigger(availableNow=True)
    .toTable(
        "fraud_detection.quarantine.realtime_transactions"
    )
)

invalid_query.awaitTermination()

# Verification
valid_count = spark.table(
    "fraud_detection.silver.realtime_transactions"
).count()

quarantine_count = spark.table(
    "fraud_detection.quarantine.realtime_transactions"
).count()

print("Real-time validation completed")
print("Valid Silver records:", valid_count)
print("Quarantined records:", quarantine_count)